# Precipitación total anual – estaciones activas IDEAM

1. Lee el Catálogo Nacional de Estaciones (`CNE_IDEAM (1).xls`, carpeta Descargas) y filtra las estaciones **activas** que miden precipitación.
2. Las divide en lotes de máx. 50 estaciones (límite del portal https://atencionciudadano.ideam.gov.co/ para series anuales, con máx. 50 años por consulta) y exporta las listas de códigos.
3. Consolida los ZIP descargados del portal en una tabla larga (estación-año) y una ancha (estación × año).

In [ ]:
import io
import zipfile
from pathlib import Path

import pandas as pd

RUTA_CNE = Path.home() / 'Downloads' / 'CNE_IDEAM (1).xls'
CARPETA_SALIDA = Path('salidas_ideam_precip_anual')
CARPETA_ZIPS = CARPETA_SALIDA / 'zips'  # aquí se guardan los ZIP descargados del portal

MAX_ESTACIONES = 50  # límite del portal para series anuales

ESTADOS = ['Activa']  # agregar 'En Mantenimiento' si se quieren incluir
# Categorías hidrométricas (nivel de ríos) y radiosonda: no miden precipitación
CATEGORIAS_EXCLUIDAS = ['Limnimétrica', 'Limnigráfica', 'Radio Sonda']

CARPETA_ZIPS.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Estaciones activas del CNE ---
cne = pd.read_excel(RUTA_CNE, dtype={'CODIGO': str})
for col in ['LATITUD', 'LONGITUD']:
    cne[col] = pd.to_numeric(cne[col].astype(str).str.replace(',', '.'), errors='coerce')

estaciones = cne[cne['ESTADO'].isin(ESTADOS) & ~cne['CATEGORIA'].isin(CATEGORIAS_EXCLUIDAS)].copy()
estaciones = estaciones.sort_values('CODIGO').reset_index(drop=True)
estaciones['LOTE'] = estaciones.index // MAX_ESTACIONES + 1

estaciones.to_csv(CARPETA_SALIDA / 'estaciones_activas_precipitacion.csv', index=False, encoding='utf-8-sig')
with open(CARPETA_SALIDA / 'lotes_codigos.txt', 'w', encoding='utf-8') as f:
    for lote, g in estaciones.groupby('LOTE'):
        f.write(f'Lote {lote}: ' + ', '.join(g['CODIGO']) + '\n')

print(f'Estaciones en el CNE: {len(cne)}')
print(f'Estaciones seleccionadas: {len(estaciones)} en {estaciones.LOTE.max()} lotes')
estaciones['CATEGORIA'].value_counts()

In [ ]:
# --- Consolidación de los ZIP descargados del portal ---
def leer_csv(raw):
    for enc in ('utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(io.BytesIO(raw), sep=None, engine='python', encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue

tablas = []
for zp in sorted(CARPETA_ZIPS.glob('*.zip')):
    with zipfile.ZipFile(zp) as z:
        for nombre in z.namelist():
            if nombre.lower().endswith('.csv'):
                tablas.append(leer_csv(z.read(nombre)))

datos = pd.concat(tablas, ignore_index=True).drop_duplicates()
print(datos.columns.tolist())

datos['Valor'] = pd.to_numeric(datos['Valor'].str.replace(',', '.'), errors='coerce')
datos['Año'] = pd.to_datetime(datos['Fecha'], errors='coerce').dt.year
datos['CodigoEstacion'] = datos['CodigoEstacion'].str.strip()

meta = estaciones[['CODIGO', 'NOMBRE', 'CATEGORIA', 'TECNOLOGIA', 'DEPARTAMENTO', 'MUNICIPIO',
                   'ALTITUD', 'LATITUD', 'LONGITUD']].rename(columns={'CODIGO': 'CodigoEstacion'})
larga = meta.merge(datos[['CodigoEstacion', 'Año', 'Valor']], on='CodigoEstacion', how='inner')
larga = larga.rename(columns={'Valor': 'Precipitacion_total_anual_mm'})

ancha = larga.pivot_table(index=list(meta.columns), columns='Año',
                          values='Precipitacion_total_anual_mm', aggfunc='first').reset_index()
ancha.columns = [str(c) for c in ancha.columns]

larga.to_csv(CARPETA_SALIDA / 'precipitacion_total_anual_larga.csv', index=False, encoding='utf-8-sig')
ancha.to_csv(CARPETA_SALIDA / 'precipitacion_total_anual_ancha.csv', index=False, encoding='utf-8-sig')

print(f'Estaciones con datos: {larga.CodigoEstacion.nunique()} de {len(estaciones)}')
ancha.head()